In [5]:
import pandas as pd
import numpy as np
import re
import urllib.request
from collections import Counter

# 1. 데이터 다운로드 및 읽기
urllib.request.urlretrieve("https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary_more.csv", filename="news_summary_more.csv")
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

# 2. 중복 및 결측치 제거
data.drop_duplicates(subset=['text'], inplace=True)
data.dropna(axis=0, inplace=True)

# 3. 간단한 전처리 함수 (교과서 핵심 반영)
def preprocess(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence) # 특수문자 제거
    return sentence

data['text'] = data['text'].apply(preprocess)
data['headlines'] = data['headlines'].apply(preprocess)

# 4. 요약 데이터에 시작/종료 토큰 추가 (PyTorch Seq2Seq 필수)
data['headlines'] = data['headlines'].apply(lambda x: 'sos ' + x + ' eos')

print("데이터 준비 완료! 샘플 확인:")
print(data.head())

데이터 준비 완료! 샘플 확인:
                                           headlines  \
0  sos upgrad learner switches to career in ml  a...   
1  sos delhi techie wins free food from swiggy fo...   
2  sos new zealand end rohit sharmaled indias 12m...   
3  sos aegon life iterm insurance plan helps cust...   
4  sos have known hirani for yrs what if metoo cl...   

                                                text  
0  saurav kant an alumnus of upgrad and iiitbs pg...  
1  kunal shahs credit card bill payment platform ...  
2  new zealand defeated india by 8 wickets in the...  
3  with aegon life iterm insurance plan customers...  
4  speaking about the sexual harassment allegatio...  


In [6]:
# 단어 빈도수 계산
all_words = " ".join(data['text']).split() + " ".join(data['headlines']).split()
word_counts = Counter(all_words)

# 빈도수 낮은 단어 제외하고 사전 만들기
vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
for word, count in word_counts.items():
    if count >= 5: # 5회 이상 등장한 단어만 포함
        vocab[word] = len(vocab)

print(f"사전 크기: {len(vocab)}")

사전 크기: 37186


# Encoder

In [7]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs: 인코더의 모든 시점의 출력 (Attention에서 사용)
        # hidden, cell: 마지막 시점의 상태
        return outputs, hidden, cell

# Attention

In [8]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.v = nn.Linear(hid_dim, 1, bias=False)
        self.W = nn.Linear(hid_dim * 2, hid_dim) # 인코더 output과 디코더 hidden 결합

    def forward(self, hidden, encoder_outputs):
        # hidden: [n_layers, batch, hid_dim] -> 마지막 레이어만 사용
        # encoder_outputs: [batch, seq_len, hid_dim]
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # 현재 디코더 상태를 복사해서 시퀀스 길이만큼 확장
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        
        # 에너지 계산
        energy = torch.tanh(self.W(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        
        return torch.softmax(attention, dim=1) # 각 단어에 대한 집중도(0~1)

# Decoder

In [10]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(hid_dim + emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch_size] -> 현재 타겟 단어
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        
        # 1. 어텐션 가중치 계산
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        
        # 2. 컨텍스트 벡터 생성 (가중치 x 인코더 출력)
        weighted = torch.bmm(a, encoder_outputs)
        
        # 3. RNN 입력: 단어 임베딩 + 컨텍스트 벡터
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        # 4. 최종 예측: 임베딩 + 컨텍스트 + RNN출력 모두 사용
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=2).squeeze(1))
        
        return prediction, hidden, cell

In [11]:
# 하이퍼파라미터 설정
INPUT_DIM = len(vocab)
OUTPUT_DIM = len(vocab)
EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 객체 생성
attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT, attn)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):
        # src: [batch_size, src_len], trg: [batch_size, trg_len]
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        
        input = trg[:, 0] # <SOS> 토큰
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
            
        return outputs

model = Seq2Seq(enc, dec, device).to(device)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"])

# Early Stopping (based on validation data)

In [13]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. 문장을 인덱스 시퀀스로 변환하는 함수
def sentence_to_indices(sentence, vocab):
    return [vocab.get(word, vocab["<UNK>"]) for word in sentence.split()]

# 2. 데이터셋 클래스 커스텀
class NewsSummaryDataset(Dataset):
    def __init__(self, data, vocab):
        self.src = [sentence_to_indices(text, vocab) for text in data['text']]
        self.trg = [sentence_to_indices(head, vocab) for head in data['headlines']]
        
    def __len__(self):
        return len(self.src)
    
    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.trg[idx])

# 3. 패딩 처리를 위한 collate_fn (길이가 다른 문장들을 맞춤)
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=vocab["<PAD>"])
    trg_batch = pad_sequence(trg_batch, batch_first=True, padding_value=vocab["<PAD>"])
    return src_batch, trg_batch

# 4. 데이터셋 생성 및 데이터 로더(Iterator) 정의
dataset = NewsSummaryDataset(data, vocab)

# 학습용/검증용 분리 (8:2)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# 배치 사이즈 설정 및 이터레이터 생성
BATCH_SIZE = 64
train_iterator = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_iterator = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"train_iterator 생성 완료! (배치 수: {len(train_iterator)})")

train_iterator 생성 완료! (배치 수: 1230)


In [15]:
def evaluate(model, iterator, criterion):
    model.eval() # 평가 모드: Dropout, Batch Normalization 등을 고정
    epoch_loss = 0
    
    with torch.no_grad(): # 평가 시에는 Gradient 계산을 하지 않아 메모리를 아낌
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)
            
            # 평가 시에는 teacher_forcing_ratio를 0으로 설정하여 모델이 순수하게 자기 예측만으로 문장을 만들었음.
            output = model(src, trg, teacher_forcing_ratio=0) 
            
            output_dim = output.shape[-1]
            output = output[:, 1:, :].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(output, trg)
            epoch_loss += loss.item()
            
    return epoch_loss / len(iterator)

In [16]:
import numpy as np

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        
        # output: [batch, trg_len, vocab_size] -> [batch * (trg_len-1), vocab_size]
        # trg: [batch, trg_len] -> [batch * (trg_len-1)]
        output_dim = output.shape[-1]
        output = output[:, 1:, :].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip) # 기울기 폭주 방지
        optimizer.step()
        epoch_loss += loss.item()

        if i % 100 == 0:
            print(f"  Batch {i}/{len(iterator)} | Loss: {loss.item():.4f}")
        
    return epoch_loss / len(iterator)

# Early Stopping 설정
best_valid_loss = float('inf')
patience = 3
counter = 0

for epoch in range(10): # 예시 에폭 10
    train_loss = train(model, train_iterator, optimizer, criterion, 1)
    valid_loss = evaluate(model, valid_iterator, criterion) # evaluate 함수 별도 정의 위에 있음
    
    print(f'Epoch: {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f}')
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pt')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early Stopping triggered")
            break

  Batch 0/1230 | Loss: 6.4939
  Batch 100/1230 | Loss: 6.4651
  Batch 200/1230 | Loss: 6.3767
  Batch 300/1230 | Loss: 6.3106
  Batch 400/1230 | Loss: 6.5419
  Batch 500/1230 | Loss: 6.3914
  Batch 600/1230 | Loss: 6.1911
  Batch 700/1230 | Loss: 6.3383
  Batch 800/1230 | Loss: 6.2324
  Batch 900/1230 | Loss: 6.0260
  Batch 1000/1230 | Loss: 6.0741
  Batch 1100/1230 | Loss: 6.1319
  Batch 1200/1230 | Loss: 6.0093
Epoch: 1 | Train Loss: 6.245 | Val Loss: 6.306
  Batch 0/1230 | Loss: 5.7294
  Batch 100/1230 | Loss: 5.6469
  Batch 200/1230 | Loss: 5.6075
  Batch 300/1230 | Loss: 5.6651
  Batch 400/1230 | Loss: 5.8143
  Batch 500/1230 | Loss: 5.7526
  Batch 600/1230 | Loss: 5.4230
  Batch 700/1230 | Loss: 5.8489
  Batch 800/1230 | Loss: 5.8063
  Batch 900/1230 | Loss: 5.5311
  Batch 1000/1230 | Loss: 5.4306
  Batch 1100/1230 | Loss: 5.8015
  Batch 1200/1230 | Loss: 5.3318
Epoch: 2 | Train Loss: 5.594 | Val Loss: 6.088
  Batch 0/1230 | Loss: 4.9935
  Batch 100/1230 | Loss: 5.1262
  Batch 20

# Inference

In [18]:
# 1. 인덱스를 다시 단어로 바꾸기 위한 역사전 생성
index_to_word = {v: k for k, v in vocab.items()}

def decode_sequence(model, src_tensor, vocab, max_len=20):
    model.eval()
    with torch.no_grad():
        # 인코더에 본문 입력
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # 시작 토큰 설정
        input = torch.tensor([vocab["sos"]]).to(device)
        
        decoded_words = []
        for _ in range(max_len):
            # 디코더 실행
            output, hidden, cell = model.decoder(input, hidden, cell, encoder_outputs)
            
            # 가장 확률 높은 단어 선택
            top1 = output.argmax(1)
            word = index_to_word[top1.item()]
            
            # 종료 토큰이 나오면 중단
            if word == "eos":
                break
            
            decoded_words.append(word)
            input = top1 # 현재 예측을 다음 입력으로 사용
            
    return " ".join(decoded_words)

# 2. 실제 데이터와 비교 출력하는 함수
def summarize_and_compare(model, dataset, n_samples=5):
    for i in range(n_samples):
        src, trg = dataset[i] # 원본 데이터셋에서 샘플 추출
        src_tensor = src.unsqueeze(0).to(device)
        
        # 모델 예측
        predicted_summary = decode_sequence(model, src_tensor, vocab)
        
        # 실제 정답 (sos, eos 제거하고 출력)
        actual_summary = " ".join([index_to_word[idx.item()] for idx in trg if idx not in [vocab["sos"], vocab["eos"], vocab["<PAD>"]]])
        
        # 입력 본문
        input_text = " ".join([index_to_word[idx.item()] for idx in src if idx != vocab["<PAD>"]])

        print(f"[{i+1}]")
        print(f"입력 본문: {input_text[:100]}...") 
        print(f"실제 요약: {actual_summary}")
        print(f"예측 요약: {predicted_summary}")
        print("-" * 50)

In [19]:
# 가장 좋았던 모델 가중치 불러오기
model.load_state_dict(torch.load('best_model.pt'))

# 결과 비교 출력
summarize_and_compare(model, dataset, n_samples=5)

[1]
입력 본문: saurav kant an alumnus of upgrad and iiitbs pg program in machine learning and artificial intelligen...
실제 요약: upgrad learner switches to career in ml al with 90 salary hike
예측 요약: upgrad learner switches in in in in in
--------------------------------------------------
[2]
입력 본문: kunal shahs credit card bill payment platform cred gave users a chance to win free food from swiggy ...
실제 요약: delhi techie wins free food from swiggy for one year on cred
예측 요약: tinder to to to to for for for for
--------------------------------------------------
[3]
입력 본문: new zealand defeated india by 8 wickets in the fourth odi at hamilton on thursday to win their first...
실제 요약: new zealand end rohit sharmaled indias <UNK> winning streak
예측 요약: nz beat india beat india to win streak in india
--------------------------------------------------
[4]
입력 본문: with aegon life iterm insurance plan customers can enjoy tax benefits on your premiums paid and save...
실제 요약: aegon life iterm insurance plan 

In [22]:
import pandas as pd
from summa.summarizer import summarize

# 1. 전처리 안 된 원본 데이터를 다시 로드 (포인트)
raw_data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

print("=== 추출적 요약 결과 (Summa) ===")

for i in range(5):
    text = raw_data.iloc[i]['text'] # 전처리 안 된 텍스트
    
    # ratio를 0.4 정도로 주고, 안 나오면 words를 사용하도록 설계
    summary = summarize(text, ratio=0.4)
    
    if not summary:
        summary = summarize(text, words=20)

    print(f"[{i+1}]")
    print(f"원본 텍스트: {text[:150]}...") # 마침표가 살아있는지 확인용
    print(f"추출 요약: {summary}")
    print("-" * 50)

=== 추출적 요약 결과 (Summa) ===
[1]
원본 텍스트: Saurav Kant, an alumnus of upGrad and IIIT-B's PG Program in Machine learning and Artificial Intelligence, was a Sr Systems Engineer at Infosys with a...
추출 요약: upGrad's Online Power Learning has powered 3 lakh+ careers.
--------------------------------------------------
[2]
원본 텍스트: Kunal Shah's credit card bill payment platform, CRED, gave users a chance to win free food from Swiggy for one year. Pranav Kaushik, a Delhi techie, b...
추출 요약: Users get one CRED coin per rupee of bill paid, which can be used to avail rewards from brands like Ixigo, BookMyShow, UberEats, Cult.Fit and more.
--------------------------------------------------
[3]
원본 텍스트: New Zealand defeated India by 8 wickets in the fourth ODI at Hamilton on Thursday to win their first match of the five-match ODI series. India lost an...
추출 요약: The match witnessed India getting all out for 92, their seventh lowest total in ODI cricket history.
--------------------------------------------

# [Project] 뉴스 요약봇 만들기: 추상적 요약 vs 추출적 요약

본 프로젝트는 뉴스 기사 데이터를 활용하여 딥러닝 기반의 **추상적 요약(Abstractive Summarization)**과 라이브러리 기반의 **추출적 요약(Extractive Summarization)** 두 가지 방식을 구현하고 성능을 비교 분석하였습니다.

---

## 1. 프로젝트 개요
- **데이터셋**: [News_Summary](https://github.com/sunnysai12345/News_Summary) (news_summary_more.csv)
- **프레임워크**: PyTorch
- **주요 기법**: Seq2Seq, Attention Mechanism, Summa(Extractive)

## 2. 데이터 전처리 및 Vocab 구축
- **정제**- 중복 데이터 및 결측치를 제거하여 학습 데이터의 품질을 높임.
- **정규화**- NLTK 및 정규표현식을 활용하여 축약어 등을 표준화함.
- **사전 구축**-빈도수 5회 이하의 단어를 제외하고 `<PAD>`, `<SOS>`, `<EOS>`, `<UNK>` 토큰을 포함한 커스텀 딕셔너리 사전을 구축하여 임베딩 효율을 높임.

## 3. 모델 설계 및 학습
- **Architecture**- 2-Layer LSTM Encoder-Decoder 구조에 Bahdanau Attention을 결합함.
- **Optimization**- Adam Optimizer와 CrossEntropyLoss를 사용하였으며, Gradient Clipping을 통해 기울기 폭주를 방지함.
- **Early Stopping**- 6 Epoch에서 Validation Loss의 상승을 확인하고 학습을 조기 종료하여 최적의 가중치를 보존함.

## 4. 최종 비교 및 분석 리포트 (Step 5-2)

| 비교 항목 | 추상적 요약 | 추출적 요약 |
| :--- | :--- | :--- |
| **작동 원리** | RNN+Attention이 새로운 문장을 생성함 | TextRank 알고리즘이 중요 문장을 선택함 |
| **문법 완성도** | 단어 반복(Repetition) 현상이 발생함 | 원문을 그대로 쓰므로 문법이 완벽함 |
| **핵심어 포착** | `nz beat india` 등 맥락 포착에 성공함 | 팩트 전달이 정확하고 안정적임 |
| **주요 한계** | 문법 파괴 및 할루시네이션 발생 가능 | 원본 데이터(마침표 포함) 보존이 필수적임 |

---

## 5. 프로젝트 회고 (Reflection)

### **학습 과정에서의 성과 및 문제 해결**
1. 데이터 로더(`DataLoader`), 인덱스 기반 사전(`Vocab`), 학습 루프(`Train Loop`)를 밑바닥부터 구현하는 경험을 했다. `torchtext` 라이브러리 없이 직접 딕셔너리를 관리하며 데이터의 흐름을 깊게 이해할 수 있었다
2. **에러 핸들링**- `NameError` 및 `ModuleNotFoundError` 등 환경 설정 문제를 겪었으나, 변수의 스코프를 재정의하고 필요한 라이브러리를 적절히 설치/대체하여 해결했다.
3. **학습 모니터링**- 1,230개에 달하는 배치를 학습하며 진행 상황을 실시간으로 확인하기 위해 `batch-wise logging`을 넣어 학습이 정체되지 않고 정상적으로 진행되는지 모니터링했다.

### **아쉬운 점이나 개선 방향**
1. **추상적 요약의 품질**- 모델이 `in in in`과 같이 단어를 반복하는 현상이 나타났고, 해결하기 위해서 **Beam Search** 도입이나 **Coverage Mechanism**을 적용해볼 필요가 있겠다.
2. **데이터 전처리의 중요성**- 추출적 요약 시 전처리된 데이터(특수문자 제거)를 사용했을 때 결과가 나오지 않는 문제를 통해, 알고리즘의 특성에 따라 데이터의 상태(Raw vs Preprocessed)를 다르게 관리해야 함을 깨달았다.

---

## 6. 루브릭 자체 평가
- **어텐션 메커니즘 사용**- Seq2Seq 아키텍처 내에 Attention Layer를 성공적으로 결합함.
- **결과 비교 분석**-추상적/추출적 요약의 장단점을 샘플을 통해 구체적으로 기술함.
- **성능 확보**- Early Stopping을 적용하여 과적합을 방지하고 합리적인 요약 결과를 만들어냄.